# Ingesta de publicaciones a Bronze

Lee el CSV de publicaciones del volumen, conserva los valores originales y registra el archivo de procedencia. La tabla se reemplaza en cada ejecución para que repetir la carga no duplique filas.

> Ejecutar en Databricks con acceso al volumen y permisos `CREATE`/`MODIFY` sobre `workspace.directdata_bronze`.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

# Configuración acordada en config/bronze.json.
SOURCE_PATH = "/Volumes/workspace/directdata_bronze/archivos/recepcion/publicaciones/publicaciones.csv"
TARGET_TABLE = "workspace.directdata_bronze.publicaciones"

SOURCE_SCHEMA = StructType(
    [
        StructField("plataforma", StringType(), True),
        StructField("video_id", StringType(), True),
        StructField("fecha_publicacion", StringType(), True),
        StructField("categoria", StringType(), True),
        StructField("titulo", StringType(), True),
    ]
)

Bronze mantiene los valores tal como llegan del CSV. El tipado y la limpieza de fecha y categoría corresponden a una fase posterior. `FAILFAST` detiene la lectura si el archivo no respeta el esquema declarado.

In [ ]:
publicaciones = (
    spark.read
    .option("header", True)
    .option("mode", "FAILFAST")
    .schema(SOURCE_SCHEMA)
    .csv(SOURCE_PATH)
    .withColumn("_source_file", F.input_file_name())
)

(
    publicaciones.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(TARGET_TABLE)
)

## Comprobaciones

Compara el número de filas de la lectura con la tabla Delta y confirma que la columna de procedencia quedó almacenada.

In [ ]:
resultado = spark.table(TARGET_TABLE)
assert resultado.count() == publicaciones.count(), "El conteo de Bronze no coincide con la fuente"
assert "_source_file" in resultado.columns, "Falta la trazabilidad del archivo de origen"
display(resultado)